In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType, VectorIndexType, DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn, PipelineType)

w = WorkspaceClient()

# Create the endpoint (takes a few minutes)
w.vector_search_endpoints.create_endpoint_and_wait(
    name="steam_vs", endpoint_type=EndpointType.STANDARD)

# Create the index
w.vector_search_indexes.create_index(
    name="steam_data.default.reviews_index",
    endpoint_name="steam_vs",
    primary_key="review_id",
    index_type=VectorIndexType.DELTA_SYNC,
    delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
        source_table="steam_data.default.reviews_rag",
        pipeline_type=PipelineType.TRIGGERED,
        embedding_source_columns=[EmbeddingSourceColumn(
            name="review_clean",
            embedding_model_endpoint_name="databricks-gte-large-en")])) 

In [0]:
ep = w.vector_search_endpoints.get_endpoint(endpoint_name="steam_vs")
print(ep.endpoint_status)

In [0]:
import time

for i in range(20):
    s = w.vector_search_indexes.get_index(index_name="steam_data.default.reviews_index").status
    print(i, "ready:", s.ready, "| rows:", s.indexed_row_count, "|", s.message)
    if s.ready:
        break
    time.sleep(60)